# Explorando o Delta Lake
O Delta Lake traz transações ACID para o Spark e permite recursos como versionamento de dados.

In [ ]:
from pyspark.sql import SparkSession
from delta import *

# Configuração específica para Delta Lake
builder = SparkSession.builder.appName("Estudo_Delta") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Sessão Spark com Delta Lake pronta!")

In [ ]:
# Criando dados iniciais (Versão 0)
data = [("Produto A", 10), ("Produto B", 20)]
df = spark.createDataFrame(data, ["nome", "estoque"])
df.write.format("delta").mode("overwrite").save("data/tabela_produtos")

print("Tabela criada (Versão 0)")
spark.read.format("delta").load("data/tabela_produtos").show()

In [ ]:
# Atualizando dados (Versão 1)
new_data = [("Produto A", 50)] # Aumentamos o estoque do Produto A
df_update = spark.createDataFrame(new_data, ["nome", "estoque"])
df_update.write.format("delta").mode("append").save("data/tabela_produtos")

print("Tabela atualizada (Versão 1)")

In [ ]:
# Viajando no tempo: Lendo a versão original (0)
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load("data/tabela_produtos")
print("Dados da Versão 0 (Original):")
df_v0.show()